# 04 — Pit Stop Statistics

Computes per-track pit stop time (seconds) under three conditions:
- **Normal** (green flag)
- **SC** (Safety Car)
- **VSC** (Virtual Safety Car)

**Method:** `PitStopTime = PitOutTime(lap N+1) - PitInTime(lap N)`  
This is the actual time spent in the pit lane, unaffected by SC/VSC lap speeds.

Output saved to `data/processed/pitstop_stats.csv`.

In [ ]:
import sys, os
sys.path.append("..")
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings("ignore")

import importlib
import src.pitstop_stats as ps
importlib.reload(ps)
from src.pitstop_stats import compute_pitstop_stats, save_pitstop_stats

pd.set_option("display.float_format", "{:.2f}".format)
print("Imports OK")

## 1. Compute pit stop statistics

In [ ]:
stats, pit_stops = compute_pitstop_stats(years=(2023, 2024, 2025))
save_pitstop_stats(stats)
print(f"\nTotal pit stops matched: {len(pit_stops):,}")
print(f"Tracks covered: {stats['EventName'].nunique()}")
print(f"Conditions found: {sorted(stats['Condition'].unique())}")

## 2. Per-track pit stop time — Normal conditions

Two columns to compare:
- **PitStopTime** — raw time spent in pit lane (same regardless of condition)
- **EffectiveLoss** — time lost relative to competitors (lower under SC/VSC because field is slower)

In [ ]:
normal = stats[stats["Condition"] == "Normal"].sort_values("MedianPitTime")
print(f"{'Track':<40} {'PitTime':>9} {'EffLoss':>9} {'N':>5}")
print("-" * 68)
for _, row in normal.iterrows():
    print(f"{row['EventName']:<40} {row['MedianPitTime']:>8.1f}s {row['MedianEffLoss']:>8.1f}s {int(row['Count']):>5}")

In [ ]:
fig = px.bar(
    normal,
    x="MedianEffLoss",
    y="EventName",
    orientation="h",
    error_x="StdEffLoss",
    title="Median Effective Pit Loss by Track — Normal Conditions (2023–2025)",
    labels={"MedianEffLoss": "Effective Pit Loss (s)", "EventName": ""},
    height=700,
    color="MedianEffLoss",
    color_continuous_scale="RdYlGn_r",
)
fig.update_layout(yaxis={"categoryorder": "total ascending"}, coloraxis_showscale=False)
fig.add_vline(
    x=normal["MedianEffLoss"].median(),
    line_dash="dash",
    annotation_text=f"Overall median {normal['MedianEffLoss'].median():.1f}s",
    annotation_position="top right",
)
fig.show()

## 3. SC vs VSC vs Normal — distribution

In [ ]:
summary = (
    pit_stops.groupby("Condition")[["PitStopTime", "EffectiveLoss"]]
    .agg(["median", "mean", "std", "count"])
    .round(2)
)
print("Overall by condition (all tracks):")
print(summary.to_string())

In [ ]:
fig = px.box(
    pit_stops,
    x="Condition",
    y="EffectiveLoss",
    color="Condition",
    color_discrete_map={"Normal": "steelblue", "SC": "orange", "VSC": "gold"},
    title="Effective Pit Loss Distribution — Normal vs SC vs VSC (all tracks, 2023–2025)",
    labels={"EffectiveLoss": "Effective Pit Loss (s)"},
    height=450,
    points="outliers",
)
fig.show()

## 4. Per-track breakdown — Normal vs SC vs VSC

In [ ]:
pivot = stats.pivot_table(index="EventName", columns="Condition", values="MedianEffLoss").reset_index()

cols = ["EventName"] + [c for c in ["Normal", "SC", "VSC"] if c in pivot.columns]
pivot = pivot[cols].sort_values("Normal")

header = f"{'Track':<40}"
for c in cols[1:]:
    header += f" {c:>9}"
print(header)
print("-" * (40 + 10 * len(cols[1:])))

for _, row in pivot.iterrows():
    line = f"{row['EventName']:<40}"
    for c in cols[1:]:
        val = row.get(c)
        line += f" {val:>8.1f}s" if pd.notna(val) else f"{'n/a':>9}"
    print(line)

## 5. SC/VSC pit stop count — how often do teams pit under neutralisation?

In [ ]:
counts = (
    pit_stops.groupby("Condition")["PitStopTime"]
    .count()
    .reset_index()
    .rename(columns={"PitStopTime": "Count"})
)
counts["Pct"] = (counts["Count"] / counts["Count"].sum() * 100).round(1)
print(counts.to_string(index=False))

fig = px.pie(
    counts,
    names="Condition",
    values="Count",
    title="Pit Stop Count by Track Condition (2023–2025)",
    color="Condition",
    color_discrete_map={"Normal": "steelblue", "SC": "orange", "VSC": "gold"},
    height=400,
)
fig.show()

## 6. Year-over-year trend — are pit stops getting faster?

In [ ]:
year_trend = (
    pit_stops[pit_stops["Condition"] == "Normal"]
    .groupby("Year")["PitStopTime"]
    .agg(Median="median", Mean="mean", Count="count")
    .reset_index()
)
print(year_trend.to_string(index=False))

fig = px.bar(
    year_trend,
    x="Year",
    y="Median",
    title="Median Pit Stop Time per Year — Normal Conditions",
    labels={"Median": "Median Pit Stop Time (s)"},
    height=380,
    text_auto=".2f",
)
fig.show()